### Import libraries and setup device

- Currently for Apple Metal, Cholesky solve is not supported in PyTorch (Frustrating? Yes)

In [ ]:
import numpy as np
import pandas as pd
import torch
import math
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import urllib.request


if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
dtype = torch.float64  #Change format for Cholesky stability

print(device)

##### Import pre-trained Lookup tables for tanh activation and corresponding informations that used to build these

In [ ]:
url = "https://raw.githubusercontent.com/RezoanoorRahman/Infinitely-Wide-Deep-Neural-Network-in-Torch/main/bilinear_tables/lookup_table_tanh.pt"

data = torch.hub.load_state_dict_from_url(url,
    map_location=torch.device(device))

F_grid = data['F_grid']
F_diag = data['F_diag']
s_vec = data['s_vec']
c_vec = data['c_vec']

##### Load required scripts, directly from GitHub

In [ ]:
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/RezoanoorRahman/Infinitely-Wide-Deep-Neural-Network-in-Torch/main/scripts/kernel_updater.py",
    "kernel_updater.py"
)

from kernel_updater import relu_nngp_kernel, tanh_nngp_kernel, tanh_nngp_kernel_square_chunked, tanh_nngp_kernel_rect_chunked

#### Load the dataset

In [ ]:
n_train = 10000  # Train data size
n_test = 10000   # Test data size
depth = 20   # L
sigma_w2 = 1.45
sigma_b2 = 0.28
noise = 1e-6     # small ridge; suggested by paper

torch.manual_seed(0)

# Load MNIST dataset

transform = transforms.ToTensor()

train_ds = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
test_ds = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

# Select subset from training and test data
X_train = train_ds.data[:n_train].reshape(n_train, -1).to(dtype=dtype) / 255.0
y_train = train_ds.targets[:n_train].long()

X_test = test_ds.data[:n_test].reshape(n_test, -1).to(dtype=dtype) / 255.0
y_test = test_ds.targets[:n_test].long()

X_train = X_train.to(device)
X_test = X_test.to(device)
y_train = y_train.to(device)
y_test = y_test.to(device)

d_in = X_train.shape[1]

# We want every input vector to have norm 1
# Hence divide them by corresponding noorms
def rescale_to_constant_norm(X, target_sq_norm):
    norms = torch.linalg.norm(X, dim=1, keepdim=True).clamp_min(1e-12)
    target_norm = math.sqrt(target_sq_norm)
    return X * (target_norm / norms)

X_train = rescale_to_constant_norm(X_train, d_in)
X_test = rescale_to_constant_norm(X_test, d_in)



# One-hot zero-mean regression labels
# correct class = 0.9, incorrect = -0.1 . Hence we have T_Train as [d_in, 10] since the number of classes equals 10
def make_targets(y, num_classes=10, dtype=torch.float64):
    T = torch.full((y.shape[0], num_classes), -0.1, dtype=dtype, device=y.device)
    T[torch.arange(y.shape[0], device=y.device), y] = 0.9
    return T

T_train = make_targets(y_train, dtype=dtype)  # shape (n_train, 10)



#### Function to evaluate performance

- We first create a function that evaluates the performance of this NNGP method for given training $D_{Train}=(X_{Train}, y_{Train})$ and test $D_{Test}=(X_{Test}, y_{Test})$ dataset with $ReLu$ as the activation function.

In [ ]:
def MNIST_trainer_relu_validation(sigma_w2, sigma_b2, noise, 
    X_train, X_test, 
    y_train, y_test):

    d_in = X_train.shape[1]

    # Train
    K_DD = relu_nngp_kernel(X_train, X_train, depth, sigma_w2, sigma_b2, d_in)

    # add noise + jitter
    K_DD = K_DD + (noise + 1e-3) * torch.eye(K_DD.shape[0], device=device)

    # Solve
    L = torch.linalg.cholesky(K_DD)

    T_train = make_targets(y_train, dtype=dtype)  # shape (n_train, 10)
    alpha = torch.cholesky_solve(T_train, L)   # (n_train, 10)

    # Test
    K_starD = relu_nngp_kernel(X_test, X_train, depth, sigma_w2, sigma_b2, d_in)

    mu_test = K_starD @ alpha # shape (n_test, 10)

    # Accuracy
    y_pred = mu_test.argmax(dim=1)
    acc = (y_pred == y_test).float().mean().item()

    return acc

- Now create a similar function for Tanh activation. The following function is identical to the previous one, except it has an extra argument 'chunk_size', indicating how many components it will update at a time for sequential update of the kernels.

In [ ]:
def MNIST_trainer_tanh_validation(sigma_w2, sigma_b2, noise, 
    X_train, X_test, 
    y_train, y_test,
    chunk_size):

    T_train = make_targets(y_train, dtype=dtype)  # shape (n_train, 10)

    d_in = X_train.shape[1]

    # TRAIN
    K_DD = tanh_nngp_kernel_square_chunked(
        X_train,
        depth,
        sigma_w2,
        sigma_b2,
        d_in,
        s_vec,
        c_vec,
        F_grid,
        F_diag,
        chunk_size=chunk_size
    )

    # add noise + jitter
    K_DD = K_DD + (noise + 1e-3) * torch.eye(K_DD.shape[0], device=device)

    # Solve
    L = torch.linalg.cholesky(K_DD + 1e-3)
    alpha = torch.cholesky_solve(T_train, L)   # (n_train, 10)

    # Test
    chunk = 256
    mu_list = []

    for i in range(0, X_test.shape[0], chunk):
        X_chunk = X_test[i:i+chunk]

        K_starD_chunk = tanh_nngp_kernel_rect_chunked(
            X_chunk,
            X_train,
            depth,
            sigma_w2,
            sigma_b2,
            d_in,
            s_vec,
            c_vec,
            F_grid,
            F_diag,
            chunk_size=chunk_size
        )

        mu_chunk = K_starD_chunk @ alpha
        mu_list.append(mu_chunk)

    mu_test = torch.cat(mu_list, dim=0)

    # Accuracy
    y_pred = mu_test.argmax(dim=1)
    acc = (y_pred == y_test).float().mean().item()

    return acc

## Hyperparameter selection for ReLu activation
### Make function for cross validation

In [ ]:
def MNIST_cv_relu_validation(
    sigma_w2,
    sigma_b2,
    noise,
    X_train,
    y_train,
    n_folds=5,
    seed=42
):
    
    np.random.seed(seed)

    n = len(X_train)

    indices = np.random.permutation(n)
    folds = np.array_split(indices, n_folds)

    misclassification_errors = []

    for k in range(n_folds):

        val_idx = folds[k]

        train_idx = np.concatenate(
            [folds[i] for i in range(n_folds) if i != k]
        )

        X_tr = X_train[train_idx]
        y_tr = y_train[train_idx]

        X_val = X_train[val_idx]
        y_val = y_train[val_idx]

        error = 1- MNIST_trainer_relu_validation(
            sigma_w2=sigma_w2,
            sigma_b2=sigma_b2,
            noise=noise,
            X_train=X_tr,
            X_test=X_val,
            y_train=y_tr,
            y_test=y_val
        )

        misclassification_errors.append(error)


    mean_error = np.mean(misclassification_errors)
    std_error = np.std(misclassification_errors)
    max_error = np.max(misclassification_errors)

    return mean_error, std_error, max_error

In [ ]:
sigma_w2s= np.arange(.5, 3.05, .25)
sigma_b2s= np.arange(.5,1.5,.3)
sigma_e2s = np.logspace(-1, -6, num=4)


CV_Table = pd.MultiIndex.from_product([sigma_w2s, sigma_b2s, sigma_e2s], 
                           names=['sigma_w2s', 'sigma_bs', 'sigma_e2s']).to_frame(index=False)

CV_Table['Mean_error_cv']=0.00000

i=0

for i in np.arange(0, CV_Table.shape[0], 1):
    hyps= CV_Table.iloc[i].values
    

    mean_error, sd_error, max_error = MNIST_cv_relu_validation(sigma_w2=hyps[0], sigma_b2=hyps[1], noise=hyps[2], 
    X_train=X_train, y_train=y_train,
    n_folds=5)
    
    CV_Table.loc[i, 'Mean_error_cv'] = mean_error
    
    #print(f"Done for {i+1}/{CV_Table.shape[0]}: sigma_w2={hyps[0]}, sigma_b2={hyps[1]},sigma_e2={hyps[2]}")


In [ ]:
Best_hyps_relu = CV_Table[CV_Table.Mean_error_cv==CV_Table.Mean_error_cv.min()].iloc[0]
Best_hyps_relu

In [ ]:
MNIST_trainer_relu_validation(sigma_w2=Best_hyps_relu.sigma_w2s, 
                              sigma_b2=Best_hyps_relu.sigma_bs, 
                              noise=Best_hyps_relu.sigma_e2s, 
    X_train=X_train, X_test=X_test, 
    y_train=y_train, y_test=y_test)

### Hyperparameter selection for Tanh activation

In [ ]:
def MNIST_cv_tanh_validation(
    sigma_w2,
    sigma_b2,
    noise,
    X_train,
    y_train,
    n_folds=5,
    seed=42
):
    
    np.random.seed(seed)

    n = len(X_train)

    indices = np.random.permutation(n)
    folds = np.array_split(indices, n_folds)

    misclassification_errors = []

    for k in range(n_folds):

        val_idx = folds[k]

        train_idx = np.concatenate(
            [folds[i] for i in range(n_folds) if i != k]
        )

        X_tr = X_train[train_idx]
        y_tr = y_train[train_idx]

        X_val = X_train[val_idx]
        y_val = y_train[val_idx]

        error = 1- MNIST_trainer_tanh_validation(
            sigma_w2=sigma_w2,
            sigma_b2=sigma_b2,
            noise=noise,
            X_train=X_tr,
            X_test=X_val,
            y_train=y_tr,
            y_test=y_val,
            chunk_size=512
        )

        misclassification_errors.append(error)


    mean_error = np.mean(misclassification_errors)
    std_error = np.std(misclassification_errors)
    max_error = np.max(misclassification_errors)

    return mean_error, std_error, max_error

In [ ]:
sigma_w2s= np.arange(.5, 3.05, .25)
sigma_b2s= np.arange(.5,1.5,.3)
sigma_e2s = np.logspace(-1, -6, num=4)


CV_Table = pd.MultiIndex.from_product([sigma_w2s, sigma_b2s, sigma_e2s], 
                           names=['sigma_w2s', 'sigma_bs', 'sigma_e2s']).to_frame(index=False)

CV_Table['Mean_error_cv']=0.00000

i=0

for i in np.arange(0, CV_Table.shape[0], 1):
    hyps= CV_Table.iloc[i].values
    

    mean_error, sd_error, max_error = MNIST_cv_tanh_validation(sigma_w2=hyps[0], sigma_b2=hyps[1], noise=hyps[2], 
    X_train=X_train, y_train=y_train,
    n_folds=5)
    
    CV_Table.loc[i, 'Mean_error_cv'] = mean_error
    
    #print(f"Done for {i+1}/{CV_Table.shape[0]}: sigma_w2={hyps[0]}, sigma_b2={hyps[1]},sigma_e2={hyps[2]}")


In [ ]:
Best_hyps_tanh = CV_Table[CV_Table.Mean_error_cv==CV_Table.Mean_error_cv.min()].iloc[0]
Best_hyps_tanh

In [ ]:
MNIST_trainer_tanh_validation(sigma_w2=Best_hyps_tanh.sigma_w2s, 
                              sigma_b2=Best_hyps_tanh.sigma_bs, 
                              noise=Best_hyps_tanh.sigma_e2s, 
    X_train=X_train, X_test=X_test, 
    y_train=y_train, y_test=y_test,
                             chunk_size=512)

#### Write a function to get predicted mean and variance

- The most impressive part about using GP is that it provides uncertainty measures. Let's write another function that provides output as mean and variance.

In [ ]:
def MNIST_relu_mean_cov(sigma_w2, sigma_b2, noise, 
    X_train, X_test, 
    y_train):

    d_in = X_train.shape[1]

    # Train
    K_DD = relu_nngp_kernel(X_train, X_train, depth, sigma_w2, sigma_b2, d_in)

    # add noise + jitter
    K_DD = K_DD + (noise + 1e-3) * torch.eye(K_DD.shape[0], device=device)

    # Solve
    L = torch.linalg.cholesky(K_DD)

    T_train = make_targets(y_train, dtype=dtype)  # shape (n_train, 10)
    alpha = torch.cholesky_solve(T_train, L)   # (n_train, 10)

    # Test
    K_starD = relu_nngp_kernel(X_test, X_train, depth, sigma_w2, sigma_b2, d_in)

    mu_test = K_starD @ alpha # shape (n_test, 10)

    # Get outputs and 
    y_pred = mu_test.argmax(dim=1)

    K_star_star = relu_nngp_kernel(X_test, X_test, depth, sigma_w2, sigma_b2, d_in)

    # V = torch.linalg.inv(L) @ K_starD.T

    # Don't use that cause not numerically stable and slower

    V = torch.linalg.solve_triangular(L, K_starD.T,
                                      upper=False)

    variances = torch.diag(K_star_star - (V.T @ V)) 

    return y_pred, variances

In [ ]:
y_pred_relu, var_relu =MNIST_relu_mean_cov(sigma_w2=Best_hyps_relu.sigma_w2s, 
                              sigma_b2=Best_hyps_relu.sigma_bs, 
                              noise=Best_hyps_relu.sigma_e2s, 
    X_train=X_train, X_test=X_test, 
    y_train=y_train)



In [ ]:
out_relu = pd.DataFrame({'y_test':y_test,
              'y_pred':y_pred_relu,
              'var':var_relu})

- Let us make a function that splits the variation into multiple slabs and then compute accuracy in those slabs.
- Intuition: Slabs with lower variance should have higher accuracy.

In [ ]:
def plot_accuracy_by_variance_slab(out_df, n_slabs=10):
    df = out_df.copy()

    df["correct"] = (df["y_test"] == df["y_pred"]).astype(float)

    # Sort by predictive variance
    df = df.sort_values("var", ascending=True).reset_index(drop=True)

    # Make 10 equal-sized slabs by sorted variance
    df["slab"] = pd.qcut(
        df.index,
        q=n_slabs,
        labels=[f"S{i+1}" for i in range(n_slabs)]
    )

    slab_summary = (
        df.groupby("slab", observed=True)
          .agg(
              acc=("correct", "mean"),
              mean_var=("var", "mean"),
              min_var=("var", "min"),
              max_var=("var", "max"),
              n=("correct", "size")
          )
          .reset_index()
    )

    plt.figure(figsize=(9, 5))
    plt.bar(slab_summary["slab"], slab_summary["acc"])
    plt.xlabel("Variance slab, low to high")
    plt.ylabel("Accuracy")
    plt.title("Accuracy by predictive variance slab")
    plt.ylim(0, 1)
    plt.show()

    return slab_summary

In [ ]:
plot_accuracy_by_variance_slab(out_relu, n_slabs=15)